> **Ramya Sree Kasula**: notebook skeleton (title, architecture, setup) and Section 6 (design decisions). Teammates add their sections between them.

# PerfectPetClub – Customer Churn ETL Pipeline

**Module:** Data Engineering Team Presentation &nbsp;|&nbsp; **Notebook 1 of 2:** ETL pipeline

**Goal:** build a clean, one-row-per-customer dataset for analysing **customer churn**, where churn means a customer has **cancelled the auto-renew** of their subscription (`subscription.auto_renew == 0`).

| Section | Owner | What it does |
|---|---|---|
| 0. Setup & configuration | Team | Imports, paths, logging |
| 1. Extract | Bhavana Padala | Read SQLite DB, CRM CSV, server log CSV, survey CSV + question metadata |
| 2. Transform A – data quality & cleaning | Siddu Gundeboina | Data quality checks, type conversion, parsing, cleaning |
| 3. Transform B – feature engineering | Pravalika | Build customer-level features and the churn target |
| 4. Load | Pravalika | Write and verify the Parquet files |
| 5. Testing & validation | Denya Sree Adapa | Unit tests and end-to-end checks on the output |
| Repo, README, merge, HTML export | Ramya Sree Kasula & Denya Sree Adapa | Git repository, merging branches, run-all + HTML export |

**Architecture**

```
 SQLite DB ─┐
 CRM CSV  ──┤   extract_*()  ──►  raw dict  ──►  run_quality_checks() ──► clean_*()  ──►  build_*_features()
 Server log ┤                                   (DQ report)               (typed,         (one row per customer)
 Survey CSV ┘                                                              parsed)               │
                                                                                                  ▼
                                                               load_parquet()  ──►  output/customer_churn.parquet
                                                                                    output/customer_churn_model_ready.parquet
```

## 0. Setup & configuration

In [ ]:
import ast
import logging
import re
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# ---- Configuration (change paths here, not inside functions) ----
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

SOURCES = {
    "db": DATA_DIR / "PerfectPetClub.db",
    "crm": DATA_DIR / "PerfectPetClub_CRM_report_1.csv",
    "server_log": DATA_DIR / "PerfectPetClub_server_log_1.csv",
    "survey": DATA_DIR / "PerfectPetClub_survey.csv",
    "survey_questions": DATA_DIR / "PerfectPetClub_survey_questions.txt",
}
DB_TABLES = ["customer", "subscription", "subscription_type", "sale", "sale_product",
             "product", "product_category", "payment", "product_review"]

OUTPUT_FILE = OUTPUT_DIR / "customer_churn.parquet"
MODEL_READY_FILE = OUTPUT_DIR / "customer_churn_model_ready.parquet"

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
log = logging.getLogger("ppc_etl")
log.info("pandas %s | numpy %s", pd.__version__, np.__version__)

## 6. Design decisions, limitations & future improvements

**Design decisions**
- **Function-per-step architecture.** Each step is a pure function (`extract_*`, `run_quality_checks`, `clean_*`, `build_*_features`, `load_parquet`), so it can be tested on its own and re-run as `run_pipeline()`.
- **Data quality checks run before cleaning**, so the report reflects the source data. ERROR checks stop the pipeline; WARN checks are logged.
- **Customer-level grain with left joins** from the subscription table, so no customer is lost because they have no CRM contact or no reviews.
- **Rates per active month** rather than raw counts, to remove the tenure bias.
- **PII minimisation (UK GDPR)**: names, addresses, postcodes, password hashes and emails are dropped. `customer_id` is the only key.
- **Parquet** for typed, compressed, columnar storage, verified by reading it back after writing.

**Limitations**
- Small sample (221 customers, 86 churned), so differences between groups are indicative, not statistically conclusive.
- `satisfaction_score` is unreliable because its question wording duplicates the NPS question.
- Income is self-reported and banded, so the midpoint is an approximation.
- The server-log user-agent is truncated, so only device type/model can be recovered (no OS/browser).
- The dataset is a single snapshot. Churn timing (survival analysis) is not modelled.

**Future improvements**
- Move from notebooks to a scheduled pipeline (e.g. Python package + Airflow/Prefect), with incremental loads.
- Replace the hand-written checks with a framework such as *Great Expectations* or *pandera*, and move the tests to `pytest` with CI on GitHub Actions.
- Train a churn model (logistic regression / gradient boosting) on `customer_churn_model_ready.parquet`, and add survival analysis.
- Fix the duplicated survey question at source and record survey timestamps.